# Benchmark - run a two-player grid on Colab
Plays the two-player games of a grid (`grids/duel_*.yaml`: chess, Connect Four) that have not been played yet, one model at a time, and saves one record per game on Google Drive, next to the records of the textual games.
The agent is the same of the textual games (same modes), with the prompts of `duel_prompts.py`; the levels of a game are its opponents (`random`, `greedy`, Stockfish `sf<N>` / `elo<N>`, or the model itself `self-<mode>`, see `duel.py`).
If the run stops (or the runtime is deleted), run the notebook again: it resumes from the games not yet saved.

Recommended order: a pilot (`limit = 8`) to see the speed and the context sizes, then `duel_tier1.yaml`, then `duel_selfplay.yaml` (self-play, neutral prompt) and `duel_free.yaml` (the same self-play games with the free prompt, which does not structure the reasoning).

# Setup
Clones the repository (only the benchmark folder: the two-player games need no game files), installs vLLM, python-chess and Stockfish, compiles the Connect Four solver (`c4solver.py`: the oracle of the gate modes, it judges every move) and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [3]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.31.0", "pyyaml", "chess"], check=True)
    subprocess.run(["apt-get", "install", "-qq", "-y", "stockfish"], check=True) # the sf<N> / elo<N> opponents
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
import c4solver
if not c4solver.available(): # the Connect Four solver: the oracle of the gate modes, judges every move (g++, make, curl)
    c4solver.setup()
print("Working directory:", os.getcwd())

Working directory: /home/lillaro/Documents/Programmazione/LLMs-Play-Textual-Games/Code/Benchmark


In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [5]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

ModuleNotFoundError: No module named 'torch'

# What to run
- `grid`: the grid file (`grids/duel_*.yaml`). `models`: only these models of the grid (`None`: all of them, one after the other).
- `limit`: at most this many games per model (`None`: all). Use it for a pilot.
- `n_parallel`: games played at the same time. vLLM answers them together: more games, more tokens per minute, until the GPU memory is full.
  `"auto"` decides by itself: it starts a new game only if the contexts of all the games, at the size they are expected to reach, fit in the GPU memory (see `engine.ContextBudget`).
  The status line shows how full the GPU memory is expected to be. A number (e.g. `8`) plays always that many.

To change the models or their vLLM settings (e.g. a smaller `max_model_len` if a model does not fit), see `profiles.py`.

In [8]:
from records import LEGACY_RESULTS
from run import summary
from run_duel import load_duel_grid, run_duel_grid

grid = load_duel_grid("grids/duel_c4_gate.yaml") # when to think: gate modes vs e1 / r4 (see gate.py); or duel_c4_summary.yaml (keep / drop / summarize), duel_c4_surprise.yaml (gsurp), duel_c4_l1.yaml, duel_plain.yaml (no reasoning instruction), duel_selfplay.yaml (neutral prompt), duel_tier1.yaml (bots)
models = None # None: all the models of the grid, or e.g. ["Qwen/Qwen3-4B"]
limit = None # 8: the pilot (seed 1, the same 8 games of the neutral pilot); None: the whole grid
n_parallel = "auto"

if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results"
# the board grids (duel_c4_board_columns.yaml, duel_c4_board_both.yaml) replay games of the gate grid with another
# board text: give each a folder of its own, or its games look already played
board = grid["game_options"].get("connect_four", {}).get("board", "grid")
if board != "grid":
    results = results.rstrip("/") + "_board_" + board
read = [LEGACY_RESULTS] # the old experiments: their games are not played again
print("Results folder:", results)
summary(grid, results, read, models)

Results folder: results
model                                          mode  played  left  ~Mtokens left
Qwen/Qwen3-4B                                goracle      16     0            0.0
Qwen/Qwen3-4B                                 gcrit      16     0            0.0
Qwen/Qwen3-4B                                 gself      16     0            0.0
Qwen/Qwen3-4B                                    e1      16     0            0.0
Qwen/Qwen3-4B                                    r4      16     0            0.0
total: 80 played, 0 left (the token estimate uses some games already played with the same mode, models, games and prompts of the grid)


# Run
Every game is saved as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept.
If loading the second model of a run fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [ ]:
run_duel_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)

# Quick look
Per model, game, opponent and mode: average score on the board and adjudicated (win 1, draw 0.5, loss 0: the chess games stopped at `max_steps` are decided by Stockfish's evaluation of the final position, see `duel.py`), wins, draws and losses after the adjudication, games stopped at `max_steps`, illegal answers and random moves played for the agent per game, thousands of generated tokens per game. In Connect Four with the solver: blunders of the agent per game (moves that change the game value) and its thinking rate (what the gate modes choose by themselves).

The full analysis of the gate runs: `python gate_analysis.py <results folder>` (tables per mode, the gates against the solver, strength vs tokens).


In [9]:
from records import load_table

table = load_table(results)
games = table[(table.source == "benchmark-duel") & table.game.isin(grid["games"]) & table.model.isin(grid["models"])
              & table["mode"].isin(grid["modes"]) & table.seed.isin(grid["seeds"])].copy()
# records played before the adjudication existed: the result on the board
games["adjudicated_result"] = games.get("adjudicated_result", games.result).fillna(games.result)
games["adjudicated_score"] = games.adjudicated_result.map({"win": 1, "draw": 0.5, "loss": 0})
games.groupby(["model", "game", "opponent", "mode"]).agg(
    score=("score", "mean"), adjudicated=("adjudicated_score", "mean"),
    wins=("adjudicated_result", lambda r: (r == "win").sum()), draws=("adjudicated_result", lambda r: (r == "draw").sum()),
    losses=("adjudicated_result", lambda r: (r == "loss").sum()), stopped=("termination", lambda t: (t == "max_steps").sum()),
    illegal=("illegal", "mean"), forced=("forced", "mean"),
    blunders=("blunders", "mean"), think_rate=("think_rate", "mean"), # Connect Four with the solver: moves that change the game value, thinking frequency
    predicted=("predictions", "mean"), hits=("prediction_hits", "mean"), # the predict prompt / gsurp: expected replies and how many came true
    ktokens=("tokens_generated", lambda t: t.mean() / 1000)).round(2)

score  adjudicated  wins  draws  \
model         game         opponent mode                                       
Qwen/Qwen3-4B connect_four greedy   e1        0.00         0.00     0      0   
                                    gcrit     0.00         0.00     0      0   
                                    goracle   0.00         0.00     0      0   
                                    gself     0.00         0.00     0      0   
                                    r4        0.00         0.00     0      0   
                           self-e1  e1        1.00         1.00     8      0   
                                    gcrit     0.50         0.50     4      0   
                                    goracle   0.75         0.75     6      0   
                                    gself     0.25         0.25     2      0   
                                    r4        0.25         0.25     2      0   

                                             losses  stopped  illegal  forced  \
model         game         opponent mode                                        
Qwen/Qwen3-4B connect_four greedy   e1            8        0     0.00     0.0   
                                    gcrit         8        0     0.50     0.0   
                                    goracle       8        0     0.00     0.0   
                                    gself         8        0     0.00     0.0   
                                    r4            8        0     0.12     0.0   
                           self-e1  e1            0        0     0.00     0.0   
                                    gcrit         4        0     0.25     0.0   
                                    goracle       2        0     0.00     0.0   
                                    gself         6        0     0.00     0.0   
                                    r4            6        0     0.00     0.0   

                                             blunders  think_rate  ktokens  
model         game         opponent mode                                    
Qwen/Qwen3-4B connect_four greedy   e1           2.75        0.86    31.80  
                                    gcrit        2.50        0.29     4.56  
                                    goracle      2.38        0.41    15.52  
                                    gself        3.38        0.56    24.49  
                                    r4           2.88        0.17     4.55  
                           self-e1  e1           2.50        0.85    21.61  
                                    gcrit        2.00        0.34     7.61  
                                    goracle      1.12        0.43    10.77  
                                    gself        4.12        0.45    21.80  
                                    r4           3.12        0.19     5.00